# 00 · Setup (safe to run again and again)
Creates our schema, and loads the two station reference tables.

In [0]:
dbutils.widgets.text("env", "dev")
dbutils.widgets.text("catalog", "workspace")

import os, sys
sys.path.append(os.path.abspath("../src"))
from metro.config import names

print(os.getcwd())          # which folder the notebook is running in
print(os.listdir("..")) 

N = names(dbutils.widgets.get("env"), dbutils.widgets.get("catalog"))
C = N["fq"]
print("Environment:", C)

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {C}")

In [0]:
from metro.stations import STATIONS

rows = [(s["station_id"], s["name"], s["lines"][0], "/".join(s["lines"]), s["kind"],
         s["capacity_5min"], s["gates"], len(s["lines"]) > 1) for s in STATIONS]

(spark.createDataFrame(rows, "station_id STRING, station_name STRING, line STRING, lines STRING, kind STRING, "
                             "capacity_5min INT, gates INT, is_interchange BOOLEAN")
      .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{C}.dim_station"))

In [0]:
import csv

path = os.path.abspath("../reference /station_code_map.csv")   # .../metro-pulse/reference/station_code_map.csv

mapping = []                                   # start with an empty list
with open(path) as f:                          # open the file (closed automatically at the end)
    reader = csv.DictReader(f)                 # reads each line as a dictionary, using the header as keys
    for r in reader:                           # r = one row, e.g. {"source_system": "AFC", "source_code": "RDM", "station_id": "HM49", "notes": "renamed in AFC in 2025; was RDG"}
        row = (r["source_system"],             # 1. "AFC"
               r["source_code"],               # 2. "RDM"
               r["station_id"])                # 3. "HM49"   (notes is left out: it's for humans)
        mapping.append(row)                    # add this row to the list

schema = "source_system STRING, source_code STRING, station_id STRING"
df = spark.createDataFrame(mapping, schema)    # 120 rows → a table in memory (DataFrame)
w = df.write                                   # "I want to save it…"
w = w.mode("overwrite")                        # "…replacing the old table on rerun (still 120 rows)…"
w.saveAsTable(f"{C}.map_station_code")         # "…as the permanent table workspace.metro_dev.map_station_code"

In [0]:
print(spark.table(f"{C}.dim_station").count(), "stations,",
      spark.table(f"{C}.map_station_code").count(), "code mappings")
display(spark.table(f"{C}.dim_station"))